In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import FEATURES, PROCESSED_DIR, RANDOM_STATE
from src.extraction import load_raw_data
from src.preprocessing import (make_clean, prepare_clustering, run_full_pca,
                               n_components_for_threshold)
from src.clustering import (grid_kmeans, fit_kmeans, grid_dbscan, fit_dbscan,score_clustering)
from src.classification import MODELS, train_and_log, save_pipeline
from sklearn.model_selection import train_test_split

## Feature Story 1 : EDA

In [ ]:
df = load_raw_data()
print(df.shape)          # dimensions
print(df.dtypes)         # types
df.head()

### # Valeurs manquantes (nombre + %), triées par ordre décroissant

In [ ]:
manquants = df.isna().sum().to_frame("nb_manquants")
manquants["pct"] = (manquants["nb_manquants"] / len(df) * 100).round(2)
manquants = manquants.sort_values("nb_manquants", ascending=False)
display(manquants)

print("Doublons :", df.duplicated().sum())
df[FEATURES].describe()

### # Histogrammes + KDE, skewness dans le titre

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"{col}\nskew = {df[col].skew():.2f}")
for ax in axes[len(FEATURES):]:
    ax.axis("off")                      # cache la case vide (7 variables / 8 cases)
plt.tight_layout(); plt.show()

### Heatmap des corrélations

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation"); plt.show()

### Outliers

In [ ]:
# Boxplots (outliers)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
for ax in axes[len(FEATURES):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

# Nombre d'outliers
rows = []
for col in FEATURES:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    rows.append({"variable": col, "nb_outliers": n, "pct": round(n / len(df) * 100, 1)})
pd.DataFrame(rows)

## Feature Story 2 : Prétraitement

### Nettoyage minimal + sauvegarde AVANT toute transformation 

In [ ]:
df_clean = make_clean(df)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DIR / "df_clean.csv", index=False)
print(df_clean.shape)

### Branche clustering

In [ ]:
df_log, df_prepare_clustering = prepare_clustering(df_clean)
df_prepare_clustering.to_csv(PROCESSED_DIR / "df_prepare_clustering.csv", index=False)

# Histogrammes après log
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, FEATURES):
    sns.histplot(df_log[col], kde=True, ax=ax)
    ax.set_title(f"{col} (log)\nskew = {df_log[col].skew():.2f}")
for ax in axes[len(FEATURES):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

# Tableau avant / après
skew_table = pd.DataFrame({
    "skew_avant": df_clean[FEATURES].skew(),
    "skew_apres_log": df_log.skew(),
}).round(2)
skew_table

### Une vision provisoire

In [ ]:
skew_table["variation_absolue"] = (
    skew_table["skew_apres_log"].abs()
    - skew_table["skew_avant"].abs()
)

skew_table

###  PCA complète 

In [ ]:
pca, X_pca = run_full_pca(df_prepare_clustering)
var = pca.explained_variance_ratio_
cumul = np.cumsum(var)
n_80 = n_components_for_threshold(pca, 0.80)

display(pd.DataFrame({"composante": range(1, len(var) + 1),
                      "variance": var.round(3), "variance_cumulee": cumul.round(3)}))

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumul) + 1), cumul, marker="o")
plt.axhline(0.80, color="red", linestyle="--", label="seuil 80 %")
plt.axvline(n_80, color="green", linestyle="--", label=f"{n_80} composantes")
plt.xlabel("Nombre de composantes"); plt.ylabel("Variance cumulée")
plt.title("Variance expliquée cumulée (PCA)"); plt.legend(); plt.show()
print("Première estimation :", n_80, "composantes")

## Features story 3

### Grille K-means

In [ ]:
res_km = grid_kmeans(X_pca, comps=range(2, 6), ks=range(2, 8))

# Heatmap composantes x k (valeur = silhouette)
pivot = res_km.pivot(index="n_components", columns="k", values="silhouette")
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
plt.title("Silhouette : composantes PCA x k"); plt.show()

# Courbes elbow et silhouette (vs k), une ligne par nombre de composantes
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for n, g in res_km.groupby("n_components"):
    axes[0].plot(g["k"], g["inertia"], marker="o", label=f"{n} comp.")
    axes[1].plot(g["k"], g["silhouette"], marker="o", label=f"{n} comp.")
axes[0].set_title("Elbow (inertie)"); axes[1].set_title("Silhouette")
for ax in axes: ax.set_xlabel("k"); ax.legend()
plt.show()


res_km.sort_values("silhouette", ascending=False).head(10)

### Choix de la combinaison

In [ ]:
ok = res_km[res_km["min_pct"] >= 10].sort_values("silhouette", ascending=False)
display(ok.head())

BEST_N = int(ok.iloc[0]["n_components"])
BEST_K = int(ok.iloc[0]["k"])
print("Choix :", BEST_N, "composantes, k =", BEST_K)

km, X_km = fit_kmeans(X_pca, BEST_N, BEST_K)
df_clean["cluster_kmeans"] = km.labels_
df_clean["cluster_kmeans"].value_counts(normalize=True).round(3)

### Grille DBSCAN

In [ ]:
res_db = grid_dbscan(X_pca)
display(res_db.head(10))

best_db = res_db.iloc[0]
labels_db, X_db = fit_dbscan(X_pca, int(best_db["n_components"]),
                             float(best_db["eps"]), int(best_db["min_samples"]))
df_clean["cluster_dbscan"] = labels_db
df_clean["est_atypique_dbscan"] = df_clean["cluster_dbscan"] == -1

### Comparaison K-means vs DBSCAN

In [ ]:
comparaison = pd.DataFrame({
    "K-means": score_clustering(X_km, df_clean["cluster_kmeans"].values),
    "DBSCAN": score_clustering(X_db, df_clean["cluster_dbscan"].values),
}).T.round(3)
display(comparaison)

### Choix de la méthode : K-means par défaut

In [ ]:
df_clean["cluster_final"] = df_clean["cluster_kmeans"]

### Nuage de points final (PCA=2) + centroïdes K-means

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_clean["cluster_final"],
                palette="tab10", s=15, alpha=0.6)

plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c="black", marker="X", s=250, label="Centroïdes")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(title="cluster")
plt.title("Clusters finaux (plan PCA 1-2)"); plt.show()

## Feature Story 4 : Interprétation

In [ ]:
# Moyennes par cluster: vue STANDARDISÉE
moy_std = df_prepare_clustering.groupby(df_clean["cluster_final"]).mean()

plt.figure(figsize=(9, 4))
sns.heatmap(moy_std, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Moyennes par cluster (standardisées)"); plt.show()

# Moyennes par cluster : vue RÉELLE
moy_reel = df_clean.groupby("cluster_final")[FEATURES].mean().round(0)
display(moy_reel)

In [ ]:
# Taille des clusters
tailles = df_clean["cluster_final"].value_counts().sort_index()
pct = (tailles / len(df_clean) * 100).round(1)
ax = tailles.plot(kind="bar", figsize=(6, 4))
for i, (n, p) in enumerate(zip(tailles, pct)):
    ax.text(i, n, f"{n}\n({p}%)", ha="center", va="bottom")
ax.set_title("Taille des clusters"); plt.show()

# Comparaison en vraies valeurs sur les variables clés
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["CASH_ADVANCE", "PURCHASES", "PAYMENTS"]):
    moy_reel[col].plot(kind="bar", ax=ax); ax.set_title(col)
plt.tight_layout(); plt.show()

### Nommage

In [ ]:
noms_clusters = {
    0: "Clients à Risque",
    1: "Clients Premium",
    2: "Clients à Faible Activité",
}

df_clean["target"] = df_clean["cluster_final"].map(noms_clusters)


assert df_clean["target"].notna().all(), "target contient des valeurs manquantes !"
assert set(noms_clusters) == set(df_clean["cluster_final"].unique()), "un cluster a été oublié !"
print(df_clean["target"].value_counts(normalize=True).round(3))
print(df_clean["cluster_final"].value_counts(normalize=True).round(3))

df_clean.to_csv(PROCESSED_DIR / "df_clean_labeled.csv", index=False)

## Feature Story 5 : Classification

In [ ]:
# X = les 7 variables brutes. On exclut target ET toutes les colonnes de clustering.
X = df_clean[FEATURES]
y = df_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# setup_mlflow()   # = mlflow.set_experiment("classification_clients") + backend SQLite

resultats = []
for name, (clf, grid) in MODELS.items():
    print("Entraînement :", name)
    resultats.append(train_and_log(name, clf, grid, X_train, X_test, y_train, y_test))

### Tableau comparatif, trié par F1

In [ ]:
# Tableau comparatif, trié par F1
tableau = pd.DataFrame([{
        k: r[k]
        for k in [
            "name",
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "train_time_s"]}
    for r in resultats]).sort_values("f1_score", ascending=False).round(3)
display(tableau)

# Barres F1
tableau.plot(x="name", y="f1_score", kind="bar", legend=False, figsize=(7, 4), ylim=(0, 1))
plt.title("F1-score par modèle"); plt.show()

# Matrices de confusion (heatmaps annotées) pour chaque modèle
labels = sorted(y_train.unique())
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
from sklearn.metrics import confusion_matrix
for ax, r in zip(axes.flatten(), resultats):
    cm = confusion_matrix(y_test, r["y_pred"], labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(r["name"])
plt.tight_layout(); plt.show()

# Boxplot des scores de CV (stabilité)
cv_df = pd.DataFrame(
    {
        r["name"]: r["cv_scores"]
        for r in resultats
    }
)

cv_df.boxplot(figsize=(10, 5))

plt.title("Scores F1 par fold (CV)")
plt.ylabel("F1 macro")
plt.xlabel("Modèle")
plt.show()

### Meilleur modéle + sauvegarde du pipeline complet

In [ ]:
meilleur = max(resultats, key=lambda r: r["f1_score"])
print("Meilleur modèle :", meilleur["name"])
save_pipeline(meilleur["pipeline"])

# Importance des variables (Random Forest ou Régression Logistique)
for r in resultats:
    clf = r["pipeline"].named_steps["classifier"]
    if r["name"] == "Random Forest":
        imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
        imp.plot(kind="barh", title="Importance des variables (Random Forest)", figsize=(7, 4)); plt.show()
    if r["name"] == "Logistic Regression":
        coefs = pd.DataFrame(clf.coef_, index=clf.classes_, columns=FEATURES)
        sns.heatmap(coefs, annot=True, fmt=".2f", cmap="coolwarm", center=0)
        plt.title("Coefficients (Régression Logistique)"); plt.show()